<a href="https://colab.research.google.com/github/syahidmid/google-colab/blob/main/scraping/WHOIS_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pandas tqdm -q

In [ ]:
# ============================================================
# WHOIS Scraper - who.is
# Input  : CSV dengan kolom URL
# Output : CSV dengan kolom original_url, domain, created,
#          expires, updated, status
# ============================================================

# ── 1. Install dependencies ──────────────────────────────────
# !pip install requests beautifulsoup4 pandas tqdm -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse
import time
import random
from tqdm import tqdm

# ── 2. CONFIG ────────────────────────────────────────────────
INPUT_CSV        = "/content/domains.csv"      # ← ganti nama file CSV kamu
URL_COLUMN       = "URL"              # ← nama kolom URL di CSV
OUTPUT_CSV       = "whois_results.csv"
DELAY_MIN        = 3                  # detik jeda minimum antar request
DELAY_MAX        = 6                  # detik jeda maksimum antar request

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

# ── 3. HELPER FUNCTIONS ──────────────────────────────────────

def extract_domain(url: str) -> str:
    """Ambil domain bersih dari URL (tanpa www)."""
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    parsed = urlparse(url)
    domain = parsed.netloc or parsed.path
    domain = domain.replace("www.", "").strip("/")
    return domain


def scrape_whois(domain: str) -> dict:
    """
    Scrape halaman who.is/whois/<domain> dan kembalikan dict
    berisi created, updated, expires.
    """
    result = {"created": None, "updated": None, "expires": None}
    url = f"https://who.is/whois/{domain}"

    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        raise RuntimeError(f"Request gagal: {e}")

    soup = BeautifulSoup(resp.text, "html.parser")

    # Cari <div> yang mengandung "Important Dates"
    target_div = None
    for div in soup.find_all("div"):
        h2 = div.find("h2")
        if h2 and "Important Dates" in h2.get_text():
            target_div = div
            break

    if not target_div:
        raise RuntimeError("Blok 'Important Dates' tidak ditemukan di halaman")

    # Ambil semua pasangan <dt>/<dd>
    dts = target_div.find_all("dt")
    dds = target_div.find_all("dd")

    for dt, dd in zip(dts, dds):
        label = dt.get_text(strip=True).lower()
        value = dd.get_text(strip=True)
        if "created" in label:
            result["created"] = value
        elif "updated" in label:
            result["updated"] = value
        elif "expires" in label:
            result["expires"] = value

    return result


# ── 4. MAIN PROCESS ──────────────────────────────────────────

def main():
    # Load CSV
    try:
        df_input = pd.read_csv(INPUT_CSV)
    except FileNotFoundError:
        print(f"❌ File '{INPUT_CSV}' tidak ditemukan.")
        print("   Upload CSV kamu lalu sesuaikan variabel INPUT_CSV di atas.")
        return

    if URL_COLUMN not in df_input.columns:
        print(f"❌ Kolom '{URL_COLUMN}' tidak ada di CSV.")
        print(f"   Kolom yang tersedia: {list(df_input.columns)}")
        return

    rows = []
    urls = df_input[URL_COLUMN].dropna().tolist()
    print(f"✅ Total URL ditemukan: {len(urls)}\n")

    for raw_url in tqdm(urls, desc="Scraping WHOIS"):
        domain = extract_domain(raw_url)
        row = {
            "original_url": raw_url,
            "domain"      : domain,
            "created"     : None,
            "updated"     : None,
            "expires"     : None,
            "status"      : "sukses",
        }
        try:
            whois_data = scrape_whois(domain)
            row.update(whois_data)
        except Exception as e:
            row["status"] = f"gagal: {e}"
            print(f"\n⚠️  {domain} → {e}")

        rows.append(row)

        # Jeda acak agar tidak kena rate-limit
        delay = random.uniform(DELAY_MIN, DELAY_MAX)
        time.sleep(delay)

    # Simpan output
    df_out = pd.DataFrame(rows, columns=[
        "original_url", "domain", "created", "updated", "expires", "status"
    ])
    df_out.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Selesai! Hasil disimpan ke '{OUTPUT_CSV}'")
    print(df_out.head(10))


if __name__ == "__main__":
    main()